In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, f1_score, confusion_matrix
)
import random

# ── imports locaux ────────────────────────────────────────────────────────
from src.models.mctnet import MCTNet
from src.data.dataset import CropDataset, make_splits


In [ ]:
os.chdir(r"C:\Users\chahi\Desktop\CNN_Transformer_Project\crop-classification")
print(os.getcwd())

In [ ]:
# ── reproductibilité ──────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
# ── config ────────────────────────────────────────────────────────────────
import yaml
with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
STATE      = cfg["data"]["state"]           # 'arkansas' ou 'california'
BANDS_PATH = cfg["data"]["bands_path"]
MASKS_PATH = cfg["data"]["masks_path"]
LABELS_PATH= cfg["data"]["labels_path"]

print(f"Device : {DEVICE}")
print(f"État   : {STATE}")

In [ ]:
# ── chargement labels pour les splits ────────────────────────────────────
from src.data.dataset import CDL_TO_IDX_ARK, CDL_TO_IDX_CAL
labels_raw = np.load(LABELS_PATH)
cdl_map    = CDL_TO_IDX_ARK if STATE == 'arkansas' else CDL_TO_IDX_CAL
labels     = np.array([cdl_map.get(int(l), cdl_map[-1]) for l in labels_raw])

# ── splits ────────────────────────────────────────────────────────────────
train_idx, val_idx, test_idx = make_splits(
    n_total=len(labels),
    n_per_class=cfg["data"]["n_per_class"],
    val_ratio=cfg["data"]["val_ratio"],
    labels=labels,
    seed=SEED,
)
print(f"Train : {len(train_idx)} | Val : {len(val_idx)} | Test : {len(test_idx)}")

# sauvegarde des splits pour reproductibilité
splits = {
    "train": train_idx.tolist(),
    "val":   val_idx.tolist(),
    "test":  test_idx.tolist(),
}
with open(f"outputs/{STATE}_splits.json", "w") as f:
    json.dump(splits, f)

# ── datasets & dataloaders ────────────────────────────────────────────────
dataset_kwargs = dict(
    bands_path=BANDS_PATH,
    masks_path=MASKS_PATH,
    labels_path=LABELS_PATH,
    state=STATE,
    normalize=True,
)

train_dataset = CropDataset(**dataset_kwargs, indices=train_idx)
val_dataset   = CropDataset(**dataset_kwargs, indices=val_idx)
test_dataset  = CropDataset(**dataset_kwargs, indices=test_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg["training"]["batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
val_loader = DataLoader(val_dataset,  batch_size=256, shuffle=False)
test_loader= DataLoader(test_dataset, batch_size=256, shuffle=False)
print(f"Train batches : {len(train_loader)} | Val batches : {len(val_loader)} | Test batches : {len(test_loader)}")

In [ ]:
# ── modèle ────────────────────────────────────────────────────────────────
n_classes = len(set(labels))
model = MCTNet(
    n_bands=cfg["model"]["n_bands"],
    n_classes=n_classes,
    n_stages=cfg["model"]["n_stages"],
    d_model=cfg["model"]["d_model"],
    n_heads=cfg["model"]["n_heads"],
    kernel_size=cfg["model"]["kernel_size"],
    dropout=cfg["model"]["dropout"],
).to(DEVICE)

print(f"\nMCTNet — {model.count_parameters():,} paramètres")

In [ ]:
# ── optimiseur & scheduler ────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg["training"]["learning_rate"],
    weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg["training"]["epochs"],
    eta_min=1e-6,
)
criterion = nn.CrossEntropyLoss()

In [ ]:
# ── fonctions utilitaires ─────────────────────────────────────────────────
def compute_metrics(y_true, y_pred):
    oa    = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    f1    = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return oa, kappa, f1

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.set_grad_enabled(train):
        for bands, masks, labels_batch in loader:
            bands  = bands.to(DEVICE)
            masks  = masks.to(DEVICE)
            labels_batch = labels_batch.to(DEVICE)

            logits = model(bands, masks)
            loss   = criterion(logits, labels_batch)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * len(labels_batch)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels_batch.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    oa, kappa, f1 = compute_metrics(all_labels, all_preds)
    return avg_loss, oa, kappa, f1

In [ ]:
# ── boucle d'entraînement ─────────────────────────────────────────────────
best_val_f1  = 0.0
patience_cnt = 0
history      = []

PATIENCE = cfg["training"]["early_stopping_patience"]
EPOCHS   = cfg["training"]["epochs"]
CKPT_DIR = cfg["output"]["checkpoint_dir"]
os.makedirs(CKPT_DIR, exist_ok=True)

print(f"\n{'─'*60}")
print(f"  Entraînement — {EPOCHS} epochs max, patience={PATIENCE}")
print(f"{'─'*60}")
print(f"{'Epoch':>6} {'Loss':>8} {'OA':>7} {'Kappa':>7} {'F1':>7} "
      f"{'Val_Loss':>10} {'Val_OA':>8} {'Val_F1':>8}")
print(f"{'─'*60}")

for epoch in range(1, EPOCHS + 1):

    tr_loss, tr_oa, tr_kappa, tr_f1 = run_epoch(train_loader, train=True)
    vl_loss, vl_oa, vl_kappa, vl_f1 = run_epoch(val_loader,   train=False)
    scheduler.step()

    history.append({
        "epoch": epoch,
        "train_loss": tr_loss, "train_oa": tr_oa,
        "train_f1": tr_f1,
        "val_loss": vl_loss,   "val_oa": vl_oa,
        "val_f1": vl_f1,       "val_kappa": vl_kappa,
    })

    print(f"{epoch:>6} {tr_loss:>8.4f} {tr_oa:>7.4f} {tr_kappa:>7.4f} "
          f"{tr_f1:>7.4f} {vl_loss:>10.4f} {vl_oa:>8.4f} {vl_f1:>8.4f}")

    # sauvegarde meilleur modèle (critère : val F1)
    if vl_f1 > best_val_f1:
        best_val_f1  = vl_f1
        patience_cnt = 0
        torch.save({
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "optimizer":   optimizer.state_dict(),
            "val_f1":      vl_f1,
            "val_oa":      vl_oa,
            "val_kappa":   vl_kappa,
            "config":      cfg,
        }, f"{CKPT_DIR}/{STATE}_best.pt")
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"\n  Early stopping à l'epoch {epoch}")
            break

# sauvegarde historique
with open(f"outputs/{STATE}_history.json", "w") as f:
    json.dump(history, f, indent=2)

In [ ]:
# ── évaluation finale sur le test set ────────────────────────────────────
print(f"\n{'─'*60}")
print("  Évaluation finale — Test set")
print(f"{'─'*60}")

# charge le meilleur checkpoint
ckpt = torch.load(f"{CKPT_DIR}/{STATE}_best.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])

_, test_oa, test_kappa, test_f1 = run_epoch(test_loader, train=False)

print(f"  OA    : {test_oa:.4f}")
print(f"  Kappa : {test_kappa:.4f}")
print(f"  F1    : {test_f1:.4f}")

# matrice de confusion
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for bands, masks, lbls in test_loader:
        preds = model(bands.to(DEVICE), masks.to(DEVICE)).argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbls.numpy())

cm = confusion_matrix(all_labels, all_preds)
print(f"\n  Matrice de confusion :\n{cm}")


In [ ]:
# sauvegarde résultats
results = {
    "state":      STATE,
    "test_oa":    test_oa,
    "test_kappa": test_kappa,
    "test_f1":    test_f1,
    "best_epoch": ckpt["epoch"],
    "n_params":   model.count_parameters(),
    "confusion_matrix": cm.tolist(),
}
with open(f"outputs/{STATE}_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\n  Résultats sauvegardés dans outputs/{STATE}_results.json")